# Data Ingestion and Basic Validation Pipeline Demo

This notebook demonstrates a prototype ingestion and validation pipeline using `ingestion_validation_pipeline.py`.

The workflow covers:

1. Loading a raw CSV dataset.
2. Running validation checks.
3. Generating `validated_data.csv` when validation passes.
4. Simulating a validation failure and reviewing the logged issues.

Dataset used: a small tabular sample based on the public Palmer Penguins dataset.

## 1. Import dependencies and inspect project files

In [1]:
from pathlib import Path
import pandas as pd

from ingestion_validation_pipeline import run_pipeline

DATA_DIR = Path('data')
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Project files:')
for path in sorted(Path('.').glob('*')):
    print('-', path)

Project files:
- README.md
- __pycache__
- data
- ingestion_validation_pipeline.py
- output
- pipeline_demo.ipynb
- requirements.txt
- task_comment.txt


## 2. Preview the raw sample dataset

In [2]:
raw_path = DATA_DIR / 'sample_raw_penguins.csv'
raw_df = pd.read_csv(raw_path)
raw_df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181,3750,Male,2007
1,Adelie,Torgersen,39.5,17.4,186,3800,Female,2007
2,Adelie,Torgersen,40.3,18.0,195,3250,Female,2007
3,Adelie,Torgersen,36.7,19.3,193,3450,Female,2007
4,Adelie,Torgersen,39.3,20.6,190,3650,Male,2007


## 3. Successful validation run

This scenario uses the valid sample raw dataset. Since all validation checks pass, the pipeline writes a cleaned and validated file to `output/validated_data.csv`.

In [3]:
success_result = run_pipeline(
    input_path=str(raw_path),
    output_path='output/validated_data.csv',
    log_file='output/validation_success.log',
    simulate_failure=False
)

success_result

2026-06-11 14:49:49,709 | INFO | Pipeline started.


2026-06-11 14:49:49,710 | INFO | Input path: data/sample_raw_penguins.csv


2026-06-11 14:49:49,715 | INFO | Starting validation checks...


2026-06-11 14:49:49,716 | INFO | Rows ingested: 12 | Columns ingested: 8


2026-06-11 14:49:49,717 | INFO | PASS: All required columns are present.


...

2026-06-11 14:49:49,748 | INFO | PASS: No exact duplicate rows found.


2026-06-11 14:49:49,762 | INFO | VALIDATION PASSED.


2026-06-11 14:49:49,779 | INFO | Cleaned/validated data written to: output/validated_data.csv


2026-06-11 14:49:49,780 | INFO | Final validated row count: 12


{'input_path': 'data/sample_raw_penguins.csv',
 'output_path': 'output/validated_data.csv',
 'row_count': 12,
 'column_count': 8,
 'validation_passed': True,
 'failures': [],
 'validated_row_count': 12}

## 4. View the generated validated data

In [4]:
validated_path = OUTPUT_DIR / 'validated_data.csv'
print('Validated file exists:', validated_path.exists())

validated_df = pd.read_csv(validated_path)
validated_df.head()

Validated file exists: True


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181,3750,Male,2007
1,Adelie,Torgersen,39.5,17.4,186,3800,Female,2007
2,Adelie,Torgersen,40.3,18.0,195,3250,Female,2007
3,Adelie,Torgersen,36.7,19.3,193,3450,Female,2007
4,Adelie,Torgersen,39.3,20.6,190,3650,Male,2007


## 5. Simulated validation failure

This run uses the same valid raw dataset but passes `simulate_failure=True`. The script intentionally injects bad records before validation. The pipeline should log failures and avoid writing a validated output file for that failed run.

In [5]:
failed_output_path = OUTPUT_DIR / 'validated_data_failed_run.csv'
if failed_output_path.exists():
    failed_output_path.unlink()

failure_result = run_pipeline(
    input_path=str(raw_path),
    output_path=str(failed_output_path),
    log_file='output/validation_failure.log',
    simulate_failure=True
)

failure_result

2026-06-11 14:49:49,820 | INFO | Pipeline started.


2026-06-11 14:49:49,821 | INFO | Input path: data/sample_raw_penguins.csv


2026-06-11 14:49:49,826 | INFO | Simulating validation failure by injecting bad records.


2026-06-11 14:49:49,830 | INFO | Starting validation checks...


2026-06-11 14:49:49,831 | INFO | Rows ingested: 13 | Columns ingested: 8


...

2026-06-11 14:49:49,872 | ERROR | 2. Column 'bill_length_mm' has 2 non-numeric value(s).


2026-06-11 14:49:49,873 | ERROR | 3. Column 'body_mass_g' has 2 value(s) outside expected range [2500, 7000].


2026-06-11 14:49:49,874 | ERROR | 4. Found 1 exact duplicate row(s).


2026-06-11 14:49:49,876 | ERROR | Validated output file was not generated.


{'input_path': 'data/sample_raw_penguins.csv',
 'output_path': 'output/validated_data_failed_run.csv',
 'row_count': 13,
 'column_count': 8,
 'validation_passed': False,
 'failures': ["Missing values found in critical columns: {'species': 2}",
  "Column 'bill_length_mm' has 2 non-numeric value(s).",
  "Column 'body_mass_g' has 2 value(s) outside expected range [2500, 7000].",
  'Found 1 exact duplicate row(s).']}

## 6. Review validation failure messages

In [6]:
print('Validation passed:', failure_result['validation_passed'])
print('\nFailures found:')
for issue in failure_result['failures']:
    print('-', issue)

failed_output_path = OUTPUT_DIR / 'validated_data_failed_run.csv'
print('\nFailed-run output file exists:', failed_output_path.exists())

Validation passed: False

Failures found:
- Missing values found in critical columns: {'species': 2}
- Column 'bill_length_mm' has 2 non-numeric value(s).
- Column 'body_mass_g' has 2 value(s) outside expected range [2500, 7000].
- Found 1 exact duplicate row(s).

Failed-run output file exists: False


## 7. Optional: Run validation against the included invalid dataset

The repository also includes `data/sample_invalid_penguins.csv`, which contains duplicate rows, invalid categories, missing values, incorrect data types, and out-of-range values.

In [7]:
invalid_output_path = OUTPUT_DIR / 'validated_data_from_invalid_file.csv'
if invalid_output_path.exists():
    invalid_output_path.unlink()

invalid_path = DATA_DIR / 'sample_invalid_penguins.csv'
invalid_result = run_pipeline(
    input_path=str(invalid_path),
    output_path=str(invalid_output_path),
    log_file='output/validation_invalid_file.log',
    simulate_failure=False
)

invalid_result

2026-06-11 14:49:49,902 | INFO | Pipeline started.


2026-06-11 14:49:49,903 | INFO | Input path: data/sample_invalid_penguins.csv


2026-06-11 14:49:49,908 | INFO | Starting validation checks...


2026-06-11 14:49:49,909 | INFO | Rows ingested: 5 | Columns ingested: 8


2026-06-11 14:49:49,911 | INFO | PASS: All required columns are present.


...

2026-06-11 14:49:49,936 | ERROR | 7. Column 'island' has invalid value(s): ['UnknownIsland']


2026-06-11 14:49:49,937 | ERROR | 8. Column 'sex' has invalid value(s): ['Unknown']


2026-06-11 14:49:49,938 | ERROR | 9. Found 1 exact duplicate row(s).


2026-06-11 14:49:49,939 | ERROR | Validated output file was not generated.


{'input_path': 'data/sample_invalid_penguins.csv',
 'output_path': 'output/validated_data_from_invalid_file.csv',
 'row_count': 5,
 'column_count': 8,
 'validation_passed': False,
 'failures': ["Missing values found in critical columns: {'species': 1, 'bill_length_mm': 1}",
  "Column 'bill_length_mm' has 1 non-numeric value(s).",
  "Column 'bill_depth_mm' has 1 value(s) outside expected range [10, 30].",
  "Column 'flipper_length_mm' has 1 value(s) outside expected range [150, 250].",
  "Column 'body_mass_g' has 1 value(s) outside expected range [2500, 7000].",
  "Column 'year' has 1 value(s) outside expected range [2007, 2009].",
  "Column 'island' has invalid value(s): ['UnknownIsland']",
  "Column 'sex' has invalid value(s): ['Unknown']",
  'Found 1 exact duplicate row(s).']}

## 8. Summary

This notebook demonstrates that the pipeline:

- Ingests raw tabular CSV data from a specified path.
- Performs multiple validation checks.
- Logs validation results.
- Generates a clean validated CSV only when checks pass.
- Demonstrates both successful and failed validation scenarios.